[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/05_production_engineering/reproducibility_config.ipynb)

# 08 . Reproducibility and Configuration

## Concept — Reproducibility & Configuration (revision notes)

**What it is**
- Reproducibility = getting the same results (ideally bit-identical) from the same code, data, config and seed; at minimum, results that are statistically consistent across runs.
- Configuration management = keeping every hyperparameter and setting in one explicit, validated, serializable object rather than scattered constants and argparse flags.

**Why it matters**
- Without it you cannot tell whether a 0.3% improvement is a real change or seed noise, nor rerun a result from last month.
- Config objects make experiments comparable, loggable (see the experiment-tracking notebook) and safe to sweep.

**When to use**
- Seed and record the seed for every experiment; report mean +/- std over several seeds for final claims.
- Use deterministic algorithms when debugging or when exact reproduction is required (accepting some slowdown).
- Use typed configs (dataclasses/Pydantic) for small projects, Hydra/OmegaConf when you need composition and sweeps.

**Math & intuition**
- Randomness sources: Python `random`, NumPy, torch CPU and CUDA generators, DataLoader worker processes, cuDNN autotuning, and non-deterministic GPU kernels (atomics).
- Floating point is not associative: (a + b) + c != a + (b + c). Any change in summation order (threads, kernels, hardware) changes the last bits.
- Bitwise reproducibility holds only for the same hardware, library versions and settings; across machines aim for statistical reproducibility.

### 1. The sources of randomness: what one seed does and doesn't control

**What this cell does (plain language):** we draw a random number from each of Python's `random`, NumPy and torch after seeding only one of them. Seeding Python's generator does nothing for torch and vice versa, so a proper `seed_everything` must seed all of them. We then run it twice to prove outputs repeat.

In [1]:
import os, random, json, hashlib, platform, tempfile, copy, dataclasses
import numpy as np, torch, torch.nn as nn

def draw():
    return round(random.random(), 6), round(float(np.random.rand()), 6), round(torch.rand(1).item(), 6)

random.seed(0)                                   # seeds ONLY python's generator
a = draw(); random.seed(0); b = draw()
print("seed only python  ->", a, "then", b)       # python value repeats, numpy/torch values differ

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)   # torch.manual_seed also seeds CUDA
    os.environ["PYTHONHASHSEED"] = str(seed)      # only effective if set BEFORE the interpreter starts
    return seed

seed_everything(0); a = draw(); seed_everything(0); b = draw()
print("seed everything   ->", a, "then", b, "| identical:", a == b)

seed only python  -> (0.844422, 0.166799, 0.734426) then (0.844422, 0.164167, 0.380376)
seed everything   -> (0.844422, 0.548814, 0.496257) then (0.844422, 0.548814, 0.496257) | identical: True


### 2. Local generators: randomness that doesn't interfere

**What this cell does (plain language):** relying on the global RNG means any library call that consumes random numbers shifts every later draw. A `torch.Generator` object gives a private stream. We show that inserting an extra `torch.rand` call changes global-RNG results downstream but leaves a dedicated generator's stream untouched.

In [2]:
def pipeline(extra_draw):
    torch.manual_seed(0)
    g = torch.Generator().manual_seed(1)               # private stream for e.g. data shuffling
    if extra_draw: torch.rand(3)                        # some unrelated code consumes global randomness
    return torch.rand(2).tolist(), torch.rand(2, generator=g).tolist()

(g1, p1), (g2, p2) = pipeline(False), pipeline(True)
print("global-RNG draws equal :", g1 == g2)             # False: shifted by the extra draw
print("private-generator equal:", p1 == p2)             # True

global-RNG draws equal : False
private-generator equal: True


### 3. Floating point is not associative

**What this cell does (plain language):** a big reason parallel code is not bit-reproducible: summing the same numbers in a different order gives slightly different results. We sum the same 100k float32 values in the original order, shuffled order, and in float64, and compare. The differences are tiny but real and can be amplified by training dynamics.

In [3]:
torch.manual_seed(0)
v = torch.randn(100_000) * torch.logspace(-3, 3, 100_000)    # wide dynamic range makes it visible
perm = torch.randperm(100_000)
s_orig, s_perm, s_64 = v.sum().item(), v[perm].sum().item(), v.double().sum().item()
print(f"original order : {s_orig:.6f}")
print(f"shuffled order : {s_perm:.6f}")
print(f"float64 'truth': {s_64:.6f}")
print("bitwise equal original vs shuffled:", s_orig == s_perm)
print("(a + b) + c == a + (b + c)?", (0.1 + 0.2) + 0.3 == 0.1 + (0.2 + 0.3))

original order : 9266.001953
shuffled order : 9266.008789
float64 'truth': 9266.009492
bitwise equal original vs shuffled: False
(a + b) + c == a + (b + c)? False


### 4. Deterministic algorithms mode

**What this cell does (plain language):** `torch.use_deterministic_algorithms(True)` makes PyTorch use deterministic kernels where they exist and raise an error where none exists. We enable it, trigger an op without a deterministic CPU implementation (`put_` with duplicate indices), show the error, then switch to `warn_only=True`. We also see that `torch.empty` memory gets filled (with NaN) so uninitialized-memory bugs become visible instead of random.

In [4]:
import warnings
print("enabled before:", torch.are_deterministic_algorithms_enabled())
torch.use_deterministic_algorithms(True)
print("enabled now   :", torch.are_deterministic_algorithms_enabled())
try:
    torch.zeros(5).put_(torch.tensor([0, 0]), torch.ones(2), accumulate=False)
except RuntimeError as e:
    print("raised:", str(e)[:75], "...")
print("torch.empty under deterministic mode (filled):", torch.empty(3).tolist())

torch.use_deterministic_algorithms(True, warn_only=True)         # warn instead of raise
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    torch.zeros(5).put_(torch.tensor([0, 0]), torch.ones(2), accumulate=False)
print("warn_only produced", len(w), "warning(s)")
torch.use_deterministic_algorithms(False)                          # restore the default
print("restored:", not torch.are_deterministic_algorithms_enabled())

enabled before: False


enabled now   : True
raised: put_ does not have a deterministic implementation, but you set 'torch.use_d ...
torch.empty under deterministic mode (filled): [nan, nan, nan]
warn_only produced 1 warning(s)
restored: True


### 5. GPU-side switches (shown, guarded)

**What this cell does (plain language):** on CUDA there are extra switches: cuDNN's autotuner (`benchmark=True`) picks algorithms by timing them, which can vary between runs; `cudnn.deterministic=True` forces fixed algorithms; and cuBLAS needs `CUBLAS_WORKSPACE_CONFIG` for deterministic matmuls. We set what is safe on CPU and print the GPU settings only if CUDA exists.

In [5]:
def make_deterministic(seed, warn_only=False):
    seed_everything(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"          # needed for deterministic cuBLAS (CUDA >= 10.2)
    torch.use_deterministic_algorithms(True, warn_only=warn_only)
    torch.backends.cudnn.benchmark = False                      # no timing-based algorithm selection
    torch.backends.cudnn.deterministic = True

make_deterministic(0, warn_only=True)
print("cudnn.benchmark:", torch.backends.cudnn.benchmark, "| cudnn.deterministic:", torch.backends.cudnn.deterministic)
print("cuda available:", torch.cuda.is_available(), "(on GPU also expect a small speed cost)")
torch.use_deterministic_algorithms(False)

cudnn.benchmark: False | cudnn.deterministic: True
cuda available: False (on GPU also expect a small speed cost)


### 6. DataLoader reproducibility: generator and worker_init_fn

**What this cell does (plain language):** shuffling uses the global RNG unless you pass `generator=`. We build a dataset whose items are random (augmentation-like) and load it with 2 worker processes. With a seeded `generator` the sample ORDER repeats; with a `worker_init_fn` that seeds `random`/numpy from `torch.initial_seed()` the per-worker augmentation randomness repeats too.

In [6]:
from torch.utils.data import Dataset, DataLoader

class NoisyDS(Dataset):
    def __len__(self): return 12
    def __getitem__(self, i):
        # augmentation-style randomness drawn inside a worker process
        return torch.tensor([float(i), np.random.rand(), random.random()])

def worker_init(worker_id):
    s = torch.initial_seed() % 2**32            # derived from base_seed + worker_id by DataLoader
    np.random.seed(s); random.seed(s)

def run(seed):
    g = torch.Generator().manual_seed(seed)
    dl = DataLoader(NoisyDS(), batch_size=4, shuffle=True, num_workers=2, generator=g, worker_init_fn=worker_init)
    return torch.cat([b for b in dl])

r1, r2, r3 = run(0), run(0), run(1)
print("same seed  -> identical batches (order and noise):", torch.equal(r1, r2))
print("other seed -> different:", not torch.equal(r1, r3))
print("order of first column with seed 0:", r1[:, 0].int().tolist())

same seed  -> identical batches (order and noise): True
other seed -> different: True
order of first column with seed 0: [5, 4, 0, 9, 7, 8, 1, 10, 3, 11, 6, 2]


### 7. A bitwise reproducibility check for a whole training run

**What this cell does (plain language):** the ultimate test: train the same tiny model twice from the same seed and compare the final weights bit for bit (via a SHA-256 hash of the parameter bytes). Then change only the seed and confirm the hash changes. The helper `weights_hash` is handy to log with each run.

In [7]:
def weights_hash(model):
    h = hashlib.sha256()
    for p in model.state_dict().values(): h.update(p.detach().cpu().numpy().tobytes())
    return h.hexdigest()[:12]

def train_run(seed, steps=60):
    seed_everything(seed)
    g = torch.Generator().manual_seed(seed + 100)
    X = torch.randn(256, 8, generator=g); y = (X[:, 0] - X[:, 1] > 0).long()
    model = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Dropout(0.2), nn.Linear(16, 2))
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    for _ in range(steps):
        idx = torch.randint(0, 256, (32,))
        opt.zero_grad(); nn.functional.cross_entropy(model(X[idx]), y[idx]).backward(); opt.step()
    return weights_hash(model)

h0a, h0b, h1 = train_run(0), train_run(0), train_run(1)
print("seed 0 run A:", h0a, "| run B:", h0b, "| identical:", h0a == h0b)
print("seed 1      :", h1, "| differs from seed 0:", h1 != h0a)

seed 0 run A: fcdf3b82eb49 | run B: fcdf3b82eb49 | identical: True
seed 1      : a37a489275ec | differs from seed 0: True


### 8. Typed configs with dataclasses

**What this cell does (plain language):** we replace loose constants with nested frozen dataclasses: a `ModelCfg`, `OptimCfg` and a top-level `Config`. `__post_init__` validates values at construction time (fail fast, not 3 hours into training), `asdict` serializes to JSON, and `dataclasses.replace` creates modified copies without mutating the original.

In [8]:
from dataclasses import dataclass, field, asdict, replace

@dataclass(frozen=True)
class ModelCfg:
    hidden: int = 64
    dropout: float = 0.1
    def __post_init__(self):
        if not 0 <= self.dropout < 1: raise ValueError(f"dropout must be in [0,1), got {self.dropout}")
        if self.hidden <= 0: raise ValueError("hidden must be positive")

@dataclass(frozen=True)
class OptimCfg:
    name: str = "adamw"
    lr: float = 3e-4
    weight_decay: float = 0.01

@dataclass(frozen=True)
class Config:
    seed: int = 0
    epochs: int = 5
    model: ModelCfg = field(default_factory=ModelCfg)
    optim: OptimCfg = field(default_factory=OptimCfg)

cfg = Config()
print(cfg)
cfg2 = replace(cfg, optim=replace(cfg.optim, lr=1e-3))        # copy-with-change (frozen => no mutation)
print("original lr:", cfg.optim.lr, "| new lr:", cfg2.optim.lr)
try:
    ModelCfg(dropout=1.5)
except ValueError as e:
    print("validation caught:", e)
try:
    cfg.seed = 3
except dataclasses.FrozenInstanceError:
    print("frozen: cannot mutate config after creation")
print(json.dumps(asdict(cfg2)))

Config(seed=0, epochs=5, model=ModelCfg(hidden=64, dropout=0.1), optim=OptimCfg(name='adamw', lr=0.0003, weight_decay=0.01))
original lr: 0.0003 | new lr: 0.001
validation caught: dropout must be in [0,1), got 1.5
frozen: cannot mutate config after creation
{"seed": 0, "epochs": 5, "model": {"hidden": 64, "dropout": 0.1}, "optim": {"name": "adamw", "lr": 0.001, "weight_decay": 0.01}}


### 9. Loading configs from YAML-like text, with dotted CLI overrides

**What this cell does (plain language):** real projects keep configs in files and override single values from the command line (`optim.lr=1e-3`). We parse a YAML document (PyYAML if installed, otherwise a tiny fallback parser for simple `key: value` nesting), convert it into our dataclasses, then apply dotted overrides with type coercion taken from the existing value's type. Unknown keys raise errors, so typos don't silently do nothing.

In [9]:
YAML_TEXT = '''
seed: 7
epochs: 3
model:
  hidden: 32
  dropout: 0.2
optim:
  name: sgd
  lr: 0.05
'''

def mini_yaml(text):
    # fallback parser: 2-space nested mappings of scalars only (NOT full YAML)
    root = {}
    stack = [(-1, root)]
    for line in text.splitlines():
        if not line.strip() or line.strip().startswith("#"): continue
        indent = len(line) - len(line.lstrip()); k, _, v = line.strip().partition(":")
        while stack[-1][0] >= indent: stack.pop()
        parent = stack[-1][1]
        if v.strip() == "":
            parent[k] = {}; stack.append((indent, parent[k]))
        else:
            val = v.strip()
            for cast in (int, float):
                try: val = cast(val); break
                except ValueError: pass
            parent[k] = val
    return root

try:
    import yaml; raw = yaml.safe_load(YAML_TEXT); print("parsed with PyYAML")
except ImportError:
    raw = mini_yaml(YAML_TEXT); print("PyYAML missing -> parsed with the mini fallback parser")
print("fallback parser agrees:", mini_yaml(YAML_TEXT) == raw)

def from_dict(d):
    return Config(seed=d.get("seed", 0), epochs=d.get("epochs", 5),
                  model=ModelCfg(**d.get("model", {})), optim=OptimCfg(**d.get("optim", {})))   # unknown keys -> TypeError

def apply_overrides(cfg, overrides):
    for ov in overrides:                                   # e.g. "optim.lr=1e-3"
        path, _, raw_val = ov.partition("="); *parents, leaf = path.split(".")
        node = cfg
        for p in parents: node = getattr(node, p)
        if not hasattr(node, leaf): raise KeyError(f"unknown config key: {path}")
        old = getattr(node, leaf)
        new = type(old)(raw_val) if not isinstance(old, bool) else raw_val.lower() == "true"
        # rebuild bottom-up because dataclasses are frozen
        updated = replace(node, **{leaf: new})
        cfg = replace(cfg, **{parents[0]: updated}) if parents else updated
    return cfg

cfg = from_dict(raw)
cfg = apply_overrides(cfg, ["optim.lr=1e-3", "epochs=10", "model.hidden=128"])
print(cfg)
try: apply_overrides(cfg, ["optim.lrr=1"])
except KeyError as e: print("typo caught:", e)
try: from_dict({"model": {"hidden": 8, "dropuot": 0.1}})
except TypeError as e: print("unknown key caught:", str(e)[:70])

parsed with PyYAML
fallback parser agrees: True
Config(seed=7, epochs=10, model=ModelCfg(hidden=128, dropout=0.2), optim=OptimCfg(name='sgd', lr=0.001, weight_decay=0.01))
typo caught: 'unknown config key: optim.lrr'
unknown key caught: ModelCfg.__init__() got an unexpected keyword argument 'dropuot'


### 10. Run manifest: config + environment + seed + result hash

**What this cell does (plain language):** we bundle everything needed to reproduce a run into one JSON 'manifest': the config, a stable config hash (for naming/deduplicating runs), library and platform versions, and the weights hash of the result. Saving this beside every checkpoint answers 'what exactly produced this model?' months later.

In [10]:
def config_hash(cfg): return hashlib.md5(json.dumps(asdict(cfg), sort_keys=True).encode()).hexdigest()[:8]

manifest = {
    "config": asdict(cfg), "config_hash": config_hash(cfg),
    "env": {"torch": torch.__version__, "numpy": np.__version__, "python": platform.python_version(),
            "platform": platform.platform(), "deterministic": torch.are_deterministic_algorithms_enabled()},
    "result_weights_hash": train_run(cfg.seed),
}
path = os.path.join(tempfile.mkdtemp(), "manifest.json")
with open(path, "w") as f: json.dump(manifest, f, indent=2)
print(open(path).read()[:600], "...")
print("same config rebuilt from YAML -> same hash:", config_hash(from_dict(mini_yaml(YAML_TEXT))) == config_hash(from_dict(raw)),
      "| changed config -> different hash:", config_hash(cfg) != config_hash(from_dict(raw)))

{
  "config": {
    "seed": 7,
    "epochs": 10,
    "model": {
      "hidden": 128,
      "dropout": 0.2
    },
    "optim": {
      "name": "sgd",
      "lr": 0.001,
      "weight_decay": 0.01
    }
  },
  "config_hash": "f0127a81",
  "env": {
    "torch": "2.14.1+cu130",
    "numpy": "2.4.6",
    "python": "3.11.15",
    "platform": "Linux-6.18.44-fc-v70-x86_64-with-glibc2.39",
    "deterministic": false
  },
  "result_weights_hash": "7af8e2e052a9"
} ...
same config rebuilt from YAML -> same hash: True | changed config -> different hash: True


### 11. Hydra / OmegaConf (guarded)

**What this cell does (plain language):** Hydra adds config composition (config groups), command-line overrides and multirun sweeps on top of OmegaConf. We try its programmatic Compose API against a throwaway config directory; if `hydra-core` is missing we print the install command and show what a typical `@hydra.main` script looks like.

In [11]:
try:
    from hydra import initialize_config_dir, compose
    from omegaconf import OmegaConf
    cdir = tempfile.mkdtemp()
    with open(os.path.join(cdir, "config.yaml"), "w") as f: f.write(YAML_TEXT)
    with initialize_config_dir(config_dir=cdir, version_base=None):
        hcfg = compose(config_name="config", overrides=["optim.lr=1e-3", "+extra=1"])
    print(OmegaConf.to_yaml(hcfg))
except ImportError:
    print("hydra-core not installed -> `pip install hydra-core`; typical usage:")
    print('''
    @hydra.main(config_path="conf", config_name="config", version_base=None)
    def main(cfg: DictConfig):
        seed_everything(cfg.seed); train(cfg)
    # python train.py optim.lr=1e-3 model.hidden=128        (override)
    # python train.py -m optim.lr=1e-2,1e-3,1e-4            (multirun sweep)
    ''')

hydra-core not installed -> `pip install hydra-core`; typical usage:

    @hydra.main(config_path="conf", config_name="config", version_base=None)
    def main(cfg: DictConfig):
        seed_everything(cfg.seed); train(cfg)
    # python train.py optim.lr=1e-3 model.hidden=128        (override)
    # python train.py -m optim.lr=1e-2,1e-3,1e-4            (multirun sweep)
    


## Common bugs

- **Seeding only `torch.manual_seed`.** Fix: Seed Python `random`, NumPy and torch together (and CUDA is covered by `torch.manual_seed`); record the seed in the run log.
- **Seeding at the top of the notebook then running cells out of order.** Fix: Seed inside the function that starts each experiment so the result does not depend on hidden cell order.
- **Assuming `shuffle=True` is reproducible.** Fix: Pass a seeded `generator=` to the DataLoader and a `worker_init_fn` that seeds any NumPy/Python randomness in workers.
- **Enabling `cudnn.benchmark=True` and expecting repeatable runs.** Fix: Set `benchmark=False`, `deterministic=True`, `torch.use_deterministic_algorithms(True)` and `CUBLAS_WORKSPACE_CONFIG` when exactness matters.
- **Believing bitwise equality across different GPUs, library versions or thread counts.** Fix: Expect only statistical reproducibility there; compare metrics over several seeds.
- **Comparing two methods using a single seed.** Fix: Run 3-5+ seeds and report mean and standard deviation; a gain smaller than seed noise is not a result.
- **Mutable global config dicts edited anywhere in the code.** Fix: Use frozen dataclasses (or OmegaConf structured configs) and pass them explicitly; create modified copies with `replace`.
- **Hyperparameters hard-coded in the script and absent from logs.** Fix: Serialize the full config into the run directory and log the config hash.
- **Silent typos in override keys (`lerning_rate=1e-3` ignored).** Fix: Validate: unknown keys must raise (dataclass constructors do; Hydra struct mode does).
- **Relying on `PYTHONHASHSEED` set from inside Python.** Fix: It must be set before the interpreter starts (environment/launcher); otherwise set iteration order can differ between runs.

## Exercises

**Exercise 1.** Write `seed_everything` variant that returns a dictionary of the three RNG states, then restores them, and show that restoring replays the same random numbers.

In [12]:
# Your attempt for Exercise 1 here

**Solution 1**

In [13]:
# Solution 1
def get_states(): return {"py": random.getstate(), "np": np.random.get_state(), "torch": torch.get_rng_state()}
def set_states(s): random.setstate(s["py"]); np.random.set_state(s["np"]); torch.set_rng_state(s["torch"])
seed_everything(0); st = get_states()
a = draw(); set_states(st); b = draw()
print(a == b)

True


**Exercise 2.** Show that float32 summation order matters: sum `torch.tensor([1e8, 1.0, -1e8])` left-to-right vs reordered `[1e8, -1e8, 1.0]`.

In [14]:
# Your attempt for Exercise 2 here

**Solution 2**

1.0 is lost when added to 1e8 in float32 (spacing there is 8), so the order changes the answer.

In [15]:
# Solution 2
a = torch.tensor([1e8, 1.0, -1e8]); b = torch.tensor([1e8, -1e8, 1.0])
s1 = (a[0] + a[1]) + a[2]; s2 = (b[0] + b[1]) + b[2]
print(s1.item(), s2.item())   # 0.0 vs 1.0

0.0 1.0


**Exercise 3.** Extend `ModelCfg` validation: `hidden` must be a multiple of 8. Show the error for `hidden=30`.

In [16]:
# Your attempt for Exercise 3 here

**Solution 3**

In [17]:
# Solution 3
@dataclass(frozen=True)
class ModelCfg8:
    hidden: int = 64
    def __post_init__(self):
        if self.hidden % 8: raise ValueError(f"hidden must be a multiple of 8, got {self.hidden}")
try: ModelCfg8(hidden=30)
except ValueError as e: print(e)

hidden must be a multiple of 8, got 30


**Exercise 4.** Use `apply_overrides` to set `model.dropout=0.5` and `seed=3`, then print the resulting config hash and confirm the original is unchanged.

In [18]:
# Your attempt for Exercise 4 here

**Solution 4**

In [19]:
# Solution 4
base = from_dict(raw)
new = apply_overrides(base, ["model.dropout=0.5", "seed=3"])
print(config_hash(base), config_hash(new), base.model.dropout, new.model.dropout)

f45047b6 7cdea1ed 0.2 0.5


**Exercise 5.** Run `train_run(0)` three times and with `torch.set_num_threads(1)`; do the hashes still match the default-thread hash on this machine?

In [20]:
# Your attempt for Exercise 5 here

**Solution 5**

Often equal for tiny models, but nothing guarantees it: different thread counts can change reduction order in larger matmuls.

In [21]:
# Solution 5
default = torch.get_num_threads()
base = train_run(0)
torch.set_num_threads(1)
one = train_run(0)
torch.set_num_threads(default)
print(base, one, base == one)

fcdf3b82eb49 fcdf3b82eb49 True
